In [ ]:
# 2023-05-05 Выгрузка корректировочной таблицы

In [7]:
#+++ версия 2025.04.21 
#+++ версия 2025.04.14 tg: Цифриум, команда 2 Машинисты, @codeup1054, @Kimutogo,@Eques8, @serin_1995, @saturnian1, @tyuop077
from importlib import reload
import sys, os 
import ipynbname


for i in ['utils_log','datetime','glob']:
    if i in sys.modules: del  sys.modules[i] 
    exec (f'from {i} import *')  
    
tm()

i = 0
for k in dir(ps):
    if ('_' not in k) and isinstance(getattr(ps, k), str) :
        print (f"{getattr(ps, k)}{k:^9}{ps._} ", end="")
        if (i := i+1)%8 == 0 : print("")



NB_PATH_NAME = ipynbname.path().name
LOG_SHEET_NAME = '02.trans'

tg(f"🚩 Start | {LOG_SHEET_NAME} ", sheet_name=LOG_SHEET_NAME, tags=NB_PATH_NAME)

tm('\n >>>')  

 *** Start at: 19:46:06 2025-04-21  ****************************************
  BBLUE     BGRAY    BGREEN    BLBLUE    BLCYAN    BLGRAY    BLGREEN   BLILAC   
 BLLBLUE  BLMAGENTA   BLRED     BLUE       BLY    BLYELLOW  BMAGENTA    BOLD    
 BORANGE    BRED       BY      BYELLOW    CYAN    DARKCYAN      E        END    
   ERR      GRAY      GREEN     LBLUE     LGRAY    LGREEN   LMAGENTA    LRED    
 MAGENTA   ORANGE    PURPLE      RED        T       TOTAL   UNDERLINE     Y     
 YELLOW      err      0:00:02.828  ₀⡄₀₀⡄₀₂.₈₂₈ 
 >>>


datetime.datetime(2025, 4, 21, 19, 46, 8, 891341)

### 01. транскрибация

In [6]:
import cv2
import numpy as np
import whisper
from docx import Document
from docx.shared import Inches
from PIL import Image
import subprocess
import glob

tm()
tg('🚩 Транскрибация ')

models = ['tiny', 'base', 'small', 'medium', 'large', 'large-v1', 'large-v2', 'large-v3']
model = 'large-v1'


def extract_audio_ffmpeg(video_file, audio_file):
    command = [
        "ffmpeg", "-y", "-i", video_file, "-vn",
        "-acodec", "pcm_s16le", "-ar", "16000", "-ac", "1", audio_file
    ]
    subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE)


def trans_video(video_file=None, model='base'):
    num_frames = 5
    root_folder = os.path.dirname(video_file) or '.'
    root_folder_sub = os.path.splitext(video_file)[0] or '.'

    audio_folder = os.path.join(root_folder, "audio")
    keyframes_folder = os.path.join(root_folder_sub, "keyframes")
    docs_folder = os.path.join(root_folder_sub, "docs")
    trans_folder = os.path.join(root_folder_sub, "transcripts")

    for folder in [keyframes_folder, audio_folder, docs_folder, trans_folder]:
        os.makedirs(folder, exist_ok=True)

    base_name = os.path.splitext(os.path.basename(video_file))[0]
    audio_file = os.path.join(audio_folder, f"{base_name}.wav")
    docx_file = os.path.join(docs_folder, f"{base_name}_{model}.docx")
    text_file = os.path.join(trans_folder, f"{base_name}_{model}.txt")

    # tm(f"🎵 01. [[{model}]] | Извлекаем аудио из {video_file}...")
    extract_audio_ffmpeg(video_file, audio_file)

    # tm(f"📝02. Транскрибация аудио...; {model}; {video_file}")
    model_whisper = whisper.load_model(model)
    result = model_whisper.transcribe(audio_file, language='ru', verbose=False)

    segments = result['segments']
    timed_text = ""
    next_timestamp = 0

    for seg in segments:
        start_sec = int(seg["start"])
        if start_sec >= next_timestamp:
            mm = start_sec // 60
            ss = start_sec % 60
            timestamp = f"[{mm:02d}:{ss:02d}]"
            timed_text += f"\n\n{timestamp}\n"
            next_timestamp += 30
        timed_text += seg["text"].strip() + " "

    with open(text_file, 'w', encoding='utf-8') as f:
        f.write(timed_text)

    # tm(f'✍04. ({len(timed_text)}) знаков в файл {text_file}')

    full_text = timed_text
    text_segments = []
    words = full_text.split()
    segment_size = max(1, len(words) // num_frames)
    for i in range(num_frames):
        start_idx = i * segment_size
        end_idx = (i + 1) * segment_size if i < num_frames - 1 else len(words)
        text_segments.append(" ".join(words[start_idx:end_idx]))

    cap = cv2.VideoCapture(video_file)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)

    doc = Document()
    doc.add_heading('Транскрипция видео с ключевыми кадрами', 0)

    for i, frame_idx in enumerate(frame_indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if ret:
            frame_filename = os.path.join(keyframes_folder, f"keyframe_{i+1:02d}.jpg")
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image = Image.fromarray(frame_rgb)
            image.save(frame_filename)
            doc.add_heading(f"Кадр {i+1}", level=2)
            doc.add_paragraph(text_segments[i])
            doc.add_picture(frame_filename, width=Inches(3))
        else:
            print(f"⚠️ Не удалось прочитать кадр {frame_idx}")

    cap.release()
    doc.save(docx_file)
    print(f"✅ {model} || Документ сохранён: {docx_file}")


_file_mask = f"lectures_2025/*.mp4"
files = glob.glob(_file_mask)

# for n in range(20):
    
tg(f'🚩 Транскрибация {n}')  # запись в гугл таблицу 

for model in models[:]:
    for n, video_file in enumerate(files[0:1], 1):
        trans_video(video_file, model)
        tg(f"{model};🌌{video_file}")  # запись в гугл таблицу

tm('>>>')
tm('🎁')


 *** Start at: 18:22:13 2025-04-21  ****************************************


100%|█████████████████████████████████████| 72.1M/72.1M [00:04<00:00, 17.7MiB/s]
C:\Users\FunFam\anaconda3\Lib\site-packages\whisper\transcribe.py:126: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
100%|██████████████████████████████████████████████████████████████████████| 45186/45186 [00:35<00:00, 1274.09frames/s]


✅ tiny || Документ сохранён: lectures_2025\Общественное движение в России 2четверь XIX в_01\docs\Общественное движение в России 2четверь XIX в_01_tiny.docx


100%|███████████████████████████████████████| 139M/139M [00:02<00:00, 49.7MiB/s]
100%|███████████████████████████████████████████████████████████████████████| 45186/45186 [00:56<00:00, 796.87frames/s]


✅ base || Документ сохранён: lectures_2025\Общественное движение в России 2четверь XIX в_01\docs\Общественное движение в России 2четверь XIX в_01_base.docx


...

100%|█████████████████████████████████████| 2.87G/2.87G [01:54<00:00, 27.0MiB/s]
100%|████████████████████████████████████████████████████████████████████████| 45186/45186 [10:46<00:00, 69.87frames/s]


✅ large-v2 || Документ сохранён: lectures_2025\Общественное движение в России 2четверь XIX в_01\docs\Общественное движение в России 2четверь XIX в_01_large-v2.docx


100%|████████████████████████████████████████████████████████████████████████| 45186/45186 [24:02<00:00, 31.32frames/s]


✅ large-v3 || Документ сохранён: lectures_2025\Общественное движение в России 2четверь XIX в_01\docs\Общественное движение в России 2четверь XIX в_01_large-v3.docx
  1:23:52.397  ₁⡄₂₃⡄₅₂.₃₉₇ >>>
         0:00  ₁⡄₂₃⡄₅₂.₃₉₇ 🎁


datetime.datetime(2025, 4, 21, 19, 46, 5, 706061)

## 02. Open AI

In [14]:
for i in ['utils_log']:
    if i in sys.modules: del  sys.modules[i] 
    exec (f'from {i} import *')  

tm( gsheet='hackaton_2025')

import os
import wave
import math
import subprocess
import tempfile
import openai

import api_keys

client = openai.OpenAI(api_key=api_keys.OPENAI_API_KEY)

def extract_audio_ffmpeg(video_file, audio_file):
    command = [
        "ffmpeg", "-y", "-i", video_file,
        "-vn", "-acodec", "pcm_s16le",
        "-ar", "16000", "-ac", "1",
        audio_file
    ]
    subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

def get_audio_duration(audio_path):
    with wave.open(audio_path, "rb") as wf:
        frames = wf.getnframes()
        rate = wf.getframerate()
        return frames / float(rate)

def split_audio_chunks(audio_path, chunk_duration=30):
    """Разделение аудио на чанки по chunk_duration секунд."""
    with wave.open(audio_path, "rb") as wf:
        framerate = wf.getframerate()
        nchannels = wf.getnchannels()
        sampwidth = wf.getsampwidth()
        nframes = wf.getnframes()

        total_duration = nframes / framerate
        num_chunks = math.ceil(total_duration / chunk_duration)

        chunks = []
        for i in range(num_chunks):
            wf.setpos(int(i * chunk_duration * framerate))
            frames = wf.readframes(int(chunk_duration * framerate))
            tmp_chunk = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
            with wave.open(tmp_chunk.name, "wb") as chunk_wf:
                chunk_wf.setnchannels(nchannels)
                chunk_wf.setsampwidth(sampwidth)
                chunk_wf.setframerate(framerate)
                chunk_wf.writeframes(frames)
            chunks.append((i * chunk_duration, tmp_chunk.name))
        return chunks

def streaming_transcription(video_file, text_file, model="whisper-1"):
    audio_file = video_file.replace(".mp4", ".wav")
    extract_audio_ffmpeg(video_file, audio_file)

    print(f"📦 Разделяем аудио на чанки...")
    chunks = split_audio_chunks(audio_file, chunk_duration=30)

    with open(text_file, 'w', encoding='utf-8') as f:
        for start_sec, chunk_path in chunks:
            with open(chunk_path, "rb") as audio:
                result = client.audio.transcriptions.create(
                    model=model,
                    file=audio,
                    response_format="verbose_json"
                )
                # Таймкод
                mm = int(start_sec) // 60
                ss = int(start_sec) % 60
                timestamp = f"[{mm:02d}:{ss:02d}]"
                
                _str = f"\n\n{timestamp}\n{result.text.strip()}"
                
                f.write(_str)
            tm(f"📝 Транскрибация [[{chunk_path}]] {_str} ", log=f"{model} {video_file} {start_sec}")
            os.remove(chunk_path)

    print(f"✅ Текст транскрибации сохранён в {text_file}")


video_file = "lectures_2025/Общественное движение в России 2четверь XIX в_01.mp4"
text_file = video_file.replace(".mp4", "_streamed.txt")
streaming_transcription(video_file, text_file)

tm('>>>', gsheet='hackaton_2025')

 *** Start at: 07:48:40 2025-04-17  ****************************************
📦 Разделяем аудио на чанки...
  0:00:04.390  ₀⡄₀₀⡄₀₄.₃₉₀ 📝 Транскрибация [[C:\Users\FunFam\AppData\Local\Temp\tmpjnacwpsl.wav]] 

[00:00]
ЕГЭ – это просто. Общественное движение в России во второй четверти девятнадцатого века В общественном движении второй четверти девятнадцатого века можно выделить три направления – консервативное, либеральное и радикальное. Идеологом консерватизма стал Сергей Семенович Уваров. Теория официальной народности, разработанная Уваровым, базировалась на трех ключевых направлениях. 
  0:00:03.422  ₀⡄₀₀⡄₀₇.₈₁₂ 📝 Транскрибация [[C:\Users\FunFam\AppData\Local\Temp\tmp2av34ker.wav]] 

[00:30]
принципах самодержавие, православие и народность. Каждый из принципов имел идеологическое обоснование. Самодержавие – это основа жизни русского общества. Православие – это ориентация человека на общественный интерес, общее благо и справедливость. Народность выражала единство народа, сплоченного вок

datetime.datetime(2025, 4, 17, 7, 49, 25, 528653)